# CSAO - Feature Engineering Pipeline

**Goal:** Create production-grade features for ranking model with:
- ✓ No data leakage (proper train/val/test split)
- ✓ Smart interaction features (co-occurrence, FP-Growth)
- ✓ Domain awareness (food delivery dynamics)
- ✓ Optimal feature set (minimal redundancy)
- ✓ Scalable & testable architecture

**Feature Categories:**
1. **User Features:** Order history, preferences, tenure, recency
2. **Item Features:** Popularity, pricing, category affinity
3. **Context Features:** Time, distance, delivery, season
4. **Interaction Features:** Co-occurrence, FP-Growth confidence, user-item affinity

In [27]:
# ============================================================================
# 1. IMPORTS & CONFIG
# ============================================================================
import pandas as pd
import numpy as np
import pickle
import json
import warnings
from datetime import datetime, timedelta
from pathlib import Path
from collections import defaultdict, Counter
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

# ============================================================================
# PATHS
# ============================================================================
BASE_DIR = Path("../data")
RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"
FEATURES_DIR = BASE_DIR / "features"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


## Section 1: Load & Validate Preprocessed Data

In [29]:
print("Loading preprocessed data...")
start = pd.Timestamp.now()

# Load main datasets
orders = pd.read_csv(PROCESSED_DIR / "orders_enriched.csv")
order_items = pd.read_csv(RAW_DIR / "order_items_v2_full.csv")
user_features_base = pd.read_csv(PROCESSED_DIR / "user_features.csv")
item_features_base = pd.read_csv(PROCESSED_DIR / "item_features.csv")

# Load co-occurrence (baseline for interaction features)
with open(PROCESSED_DIR / "cooccurrence.pkl", "rb") as f:
    cooccurrence = pickle.load(f)

# Load FP-Growth patterns (for interaction features)
try:
    with open(PROCESSED_DIR / "fpgrowth_metadata.json", "r") as f:
        fpgrowth_metadata = json.load(f)
except FileNotFoundError:
    print(" FP-Growth metadata not found, will use cooccurrence only")
    fpgrowth_metadata = {}

elapsed = (pd.Timestamp.now() - start).total_seconds()
print(f" Data loaded in {elapsed:.2f}s\n")

# ============================================================================
# VALIDATION
# ============================================================================
print(" DATA VALIDATION")
print(f"Orders:           {orders.shape}")
print(f"Order items:      {order_items.shape}")
print(f"User features:    {user_features_base.shape}")
print(f"Item features:    {item_features_base.shape}")
print(f"Co-occurrence:    {cooccurrence.shape}")
print(f"FP-Growth:        {len(fpgrowth_metadata)} segments")

# Check data quality
print(f"\n Missing values:")
print(f"  Orders:        {orders.isnull().sum().sum()}")
print(f"  Order items:   {order_items.isnull().sum().sum()}")
print(f"  User features: {user_features_base.isnull().sum().sum()}")
print(f"  Item features: {item_features_base.isnull().sum().sum()}")

# Date range
orders["order_date"] = pd.to_datetime(orders["order_date"])
print(f"\n Date range: {orders['order_date'].min()} to {orders['order_date'].max()}")

Loading preprocessed data...
 Data loaded in 2.04s

 DATA VALIDATION
Orders:           (179364, 33)
Order items:      (430612, 11)
User features:    (48640, 8)
Item features:    (2800, 4)
Co-occurrence:    (118695, 3)
FP-Growth:        9 segments

✓ Missing values:
  Orders:        0
  Order items:   0
  User features: 0
  Item features: 0

 Date range: 2019-01-01 00:00:00 to 2024-12-31 00:00:00


## Section 2: Temporal Train/Val/Test Split (NO DATA LEAKAGE)

**Strategy:** Time-based split to prevent look-ahead bias
- **Train:** Older orders (50%) → compute features from this window
- **Val:** Middle orders (25%) → test on held-out period
- **Test:** Recent orders (25%) → final evaluation

In [30]:
# ============================================================================
# TEMPORAL SPLIT (CRITICAL FOR NO DATA LEAKAGE)
# ============================================================================
orders_sorted = orders.sort_values("order_date").reset_index(drop=True)

n = len(orders_sorted)
train_end = int(0.5 * n)      # First 50%
val_end = int(0.75 * n)        # 75% (val is 25%)

train_orders = orders_sorted.iloc[:train_end].copy()
val_orders = orders_sorted.iloc[train_end:val_end].copy()
test_orders = orders_sorted.iloc[val_end:].copy()

# Mark each order with its split — match on order_id, NOT index.
# orders_sorted has reset_index(drop=True), so its .index is 0..n-1.
# The original `orders` DataFrame keeps its CSV-load index, so
# orders.index.isin(train_orders.index) would match the wrong rows silently.
orders["split"] = "test"
orders.loc[orders["order_id"].isin(train_orders["order_id"]), "split"] = "train"
orders.loc[orders["order_id"].isin(val_orders["order_id"]), "split"] = "val"

# Sanity-check — every order must be assigned exactly once
assert orders["split"].isnull().sum() == 0
assert len(orders[orders["split"] == "train"]) == len(train_orders)
assert len(orders[orders["split"] == "val"]) == len(val_orders)
assert len(orders[orders["split"] == "test"]) == len(test_orders)

print("="*80)
print("TEMPORAL TRAIN/VAL/TEST SPLIT")
print("="*80)
print(f"\nTrain: {train_orders['order_date'].min():.10} → {train_orders['order_date'].max():.10}")
print(f"  Orders: {len(train_orders):,} | Users: {train_orders['user_id'].nunique():,}")

print(f"\nVal:   {val_orders['order_date'].min():.10} → {val_orders['order_date'].max():.10}")
print(f"  Orders: {len(val_orders):,} | Users: {val_orders['user_id'].nunique():,}")

print(f"\nTest:  {test_orders['order_date'].min():.10} → {test_orders['order_date'].max():.10}")
print(f"  Orders: {len(test_orders):,} | Users: {test_orders['user_id'].nunique():,}")

print(f"\nTotal: {len(orders_sorted):,} orders")
print("="*80)

# IMPORTANT: Features will be computed FROM TRAIN DATA ONLY
# When applying to val/test, we used statistics learned from train


TEMPORAL TRAIN/VAL/TEST SPLIT

Train: .10 → .10
  Orders: 89,682 | Users: 41,733

Val:   .10 → .10
  Orders: 44,841 | Users: 29,632

Test:  .10 → .10
  Orders: 44,841 | Users: 29,630

Total: 179,364 orders


## Section 3: User Features

**Features to create:**
- `total_orders` - How many orders has user placed
- `avg_order_value` - Average spending per order
- `days_since_last_order` - Recency (engagement signal)
- `user_tenure_days` - How long user has been active
- `preferred_cuisine` - Most ordered cuisine
- `is_member` - Subscription status

In [31]:

print("Computing user features from training data...")

# Use only TRAIN orders for computing user statistics
train_order_items = order_items[order_items["order_id"].isin(train_orders["order_id"])]

user_stats = (
    train_orders
    .groupby("user_id")
    .agg(
        total_orders=("order_id", "count"),
        avg_order_value=("total_order_value", "mean"),
        max_order_value=("total_order_value", "max"),   # spec: RFM Monetary
        avg_subtotal=("subtotal", "mean"),
        min_order_date=("order_date", "min"),
        max_order_date=("order_date", "max"),
        is_member=("is_member", "first")
    )
    .reset_index()
)

# ============================================================================
# RFM METRICS (spec Section 3.4)
train_end_date = train_orders["order_date"].max()

# 1. Order frequency (orders per week) — stable across time
train_period_days = (train_end_date - train_orders["order_date"].min()).days
user_stats["order_frequency_per_week"] = (
    user_stats["total_orders"] / (train_period_days / 7 + 1)
)

# 2. Per-user short-window order counts (relative to each user's own last order date).
#    A global cutoff (fixed to train_end_date) left ~95 % of users at zero because
#    most users last ordered months/years before the end of the training window.
#    Using each user's max_order_date as the reference captures burst-ordering
#    behaviour regardless of when in the training period the user was active.
cutoff_30d = train_end_date - pd.Timedelta(days=30)  # kept for active_in_last_30d

orders_with_last = (
    train_orders[["order_id", "user_id", "order_date"]]
    .merge(user_stats[["user_id", "max_order_date"]], on="user_id")
)
orders_with_last["days_before_last"] = (
    orders_with_last["max_order_date"] - orders_with_last["order_date"]
).dt.total_seconds() / (24 * 3600)

orders_last_7d = (
    orders_with_last[orders_with_last["days_before_last"] <= 7]
    .groupby("user_id")["order_id"].count()
    .rename("orders_last_7d")
)
orders_last_30d = (
    orders_with_last[orders_with_last["days_before_last"] <= 30]
    .groupby("user_id")["order_id"].count()
    .rename("orders_last_30d")
)
user_stats = (
    user_stats
    .merge(orders_last_7d, on="user_id", how="left")
    .merge(orders_last_30d, on="user_id", how="left")
)
user_stats["orders_last_7d"]  = user_stats["orders_last_7d"].fillna(0).astype(int)
user_stats["orders_last_30d"] = user_stats["orders_last_30d"].fillna(0).astype(int)

# 3. User tenure in days — indicates loyalty
user_stats["user_tenure_days"] = (
    user_stats["max_order_date"] - user_stats["min_order_date"]
).dt.total_seconds() / (24 * 3600)

# 3b. Days since last order — proper recency signal (distance from train period end)
user_stats["days_since_last_order"] = (
    train_end_date - user_stats["max_order_date"]
).dt.total_seconds() / (24 * 3600)

# 4. Active in last 30 days of training (binary flag)
user_stats["active_in_last_30d"] = (
    user_stats["max_order_date"] >= cutoff_30d
).astype(int)

# 5. Tenure cohort (new / regular / loyal)
# include_lowest=True closes the left edge of the first bin: [0, 7] instead of
# (0, 7], so users with tenure_days=0 (single order) map to "new" instead of NaN.
# Without this, pd.cut returns NaN for 0 and .astype(str) silently produces the
# string "nan", which leaks into one-hot encoding as a spurious category.
user_stats["tenure_segment"] = pd.cut(
    user_stats["user_tenure_days"],
    bins=[0, 7, 30, float('inf')],
    labels=["new", "regular", "loyal"],
    include_lowest=True   # fix: [0, 7] closed on left so 0-day tenure → "new"
).astype(str)

# Favorite cuisine (mode)
fav_cuisine = (
    train_order_items
    .merge(train_orders[["order_id", "user_id"]], on="order_id")
    .groupby("user_id")["cuisine"]
    .apply(lambda x: x.mode()[0] if len(x.mode()) > 0 else "Unknown")
    .reset_index()
    .rename(columns={"cuisine": "preferred_cuisine"})
)

user_stats = user_stats.merge(fav_cuisine, on="user_id", how="left")

# Select features for training
user_features = user_stats[[
    "user_id",
    "total_orders",
    "avg_order_value",
    "max_order_value",
    "avg_subtotal",
    "order_frequency_per_week",
    "orders_last_7d",
    "orders_last_30d",
    "days_since_last_order",
    "user_tenure_days",
    "active_in_last_30d",
    "tenure_segment",
    "is_member",
    "preferred_cuisine"
]].copy()

print(f" User features computed: {user_features.shape}")
print(f"\n RFM metrics:")
print(f"  - order_frequency_per_week: {user_features['order_frequency_per_week'].mean():.3f} avg")
print(f"  - orders_last_7d:           {user_features['orders_last_7d'].mean():.3f} avg (per-user window)")
print(f"  - orders_last_30d:          {user_features['orders_last_30d'].mean():.3f} avg (per-user window)")
print(f"  - days_since_last_order:    {user_features['days_since_last_order'].mean():.1f} days avg")
print(f"  - active_in_last_30d:       {user_features['active_in_last_30d'].mean()*100:.1f}% of users")
print(f"  - tenure_segment:           {user_features['tenure_segment'].value_counts().to_dict()}")
print(f"\nSample user features:")
print(user_features.head(10))


Computing user features from training data...
 User features computed: (41733, 14)

 RFM metrics:
  - order_frequency_per_week: 0.014 avg
  - orders_last_7d:           1.013 avg (per-user window)
  - orders_last_30d:          1.051 avg (per-user window)
  - days_since_last_order:    394.3 days avg
  - active_in_last_30d:       5.9% of users
  - tenure_segment:           {'loyal': 25978, 'new': 15182, 'regular': 573}

Sample user features:
   user_id  total_orders  avg_order_value  max_order_value  avg_subtotal  \
0        1             4         901.2500             1295      851.2500   
1        4             4        1055.0000             1440     1000.0000   
2        5             3         896.6667             1030      836.6667   
3        8             2         707.5000             1100      707.5000   
4        9             3        1401.6667             1670     1375.0000   
5       10             5         465.0000             1010      429.0000   
6       11             1 

## Section 3b: User Behavioral Segments & Preferences

**Segment features to create:**
- `price_sensitivity` - Correlation of user AOV with item prices
- `vegetarian_pct` - Percentage of vegetarian items ordered
- `cuisine_consistency` - How varied are user's cuisine choices
- `age_group` - Demographic segment (Gen_Z, Millennial, Gen_X, Boomer)


In [32]:
# ============================================================================
# USER BEHAVIORAL SEGMENTS (FROM TRAIN SPLIT ONLY)
print("Computing user behavioral segments...")

# 1. VEGETARIAN PREFERENCE
user_veg_orders = (
    train_order_items
    .merge(train_orders[["order_id", "user_id"]], on="order_id")
    .groupby("user_id")
    .agg(
        total_items=("item_id", "count"),
        veg_items=("is_vegetarian", "sum")
    )
    .reset_index()
)

user_veg_orders["vegetarian_pct"] = (
    user_veg_orders["veg_items"] / user_veg_orders["total_items"]
).fillna(0)

# 2. PRICE SENSITIVITY (variance in AOV vs item prices)
user_price_analysis = (
    train_order_items
    .merge(train_orders[["order_id", "user_id", "total_order_value"]], on="order_id")
    .groupby("user_id")
    .agg(
        avg_item_price=("price", "mean"),
        price_std=("price", "std"),
    )
    .reset_index()
)

# Price sensitivity: users who avoid high-priced items have high sensitivity
user_price_analysis["price_sensitivity"] = (
    1 - (user_price_analysis["avg_item_price"] / train_order_items["price"].max())
).fillna(0.5)

# 3. CUISINE DIVERSITY (how many different cuisines user orders)
user_cuisine_diversity = (
    train_order_items
    .merge(train_orders[["order_id", "user_id"]], on="order_id")
    .groupby("user_id")["cuisine"]
    .nunique()
    .reset_index()
    .rename(columns={"cuisine": "cuisine_diversity"})
)

# Normalize to 0-1
max_cuisines = user_cuisine_diversity["cuisine_diversity"].max()
user_cuisine_diversity["cuisine_consistency"] = (
    1 - (user_cuisine_diversity["cuisine_diversity"] / max_cuisines)
)

# 4. AGE GROUP SEGMENTATION (get unique age per user)
def get_age_group(age):
    if age < 25:
        return "Gen_Z"
    elif age < 40:
        return "Millennial"
    elif age < 55:
        return "Gen_X"
    else:
        return "Boomer"

# Get first user_age per user (should be constant)
user_ages_series = train_orders[["user_id", "user_age"]].drop_duplicates().set_index("user_id")["user_age"]
age_groups_series = user_ages_series.apply(get_age_group)
age_groups_df = age_groups_series.reset_index()
age_groups_df.columns = ["user_id", "age_group"]

# Merge all behavioral segments - CAREFUL WITH INDEX
user_features = user_features.copy()

user_features = user_features.merge(
    user_veg_orders[["user_id", "vegetarian_pct"]],
    on="user_id",
    how="left"
)
user_features["vegetarian_pct"] = user_features["vegetarian_pct"].fillna(0)

user_features = user_features.merge(
    user_price_analysis[["user_id", "price_sensitivity"]],
    on="user_id",
    how="left"
)
user_features["price_sensitivity"] = user_features["price_sensitivity"].fillna(0.5)

user_features = user_features.merge(
    user_cuisine_diversity[["user_id", "cuisine_consistency"]],
    on="user_id",
    how="left"
)
user_features["cuisine_consistency"] = user_features["cuisine_consistency"].fillna(0)

user_features = user_features.merge(
    age_groups_df,
    on="user_id",
    how="left"
)
user_features["age_group"] = user_features["age_group"].fillna("Unknown")

print(f" User behavioral segments computed")
print(f"\n New segments:")
print(f"  - vegetarian_pct:     Mean={user_features['vegetarian_pct'].mean():.3f}")
print(f"  - price_sensitivity:  Mean={user_features['price_sensitivity'].mean():.3f}")
print(f"  - cuisine_consistency: Mean={user_features['cuisine_consistency'].mean():.3f}")
print(f"  - age_group:          {user_features['age_group'].value_counts().to_dict()}")

Computing user behavioral segments...
 User behavioral segments computed

 New segments:
  - vegetarian_pct:     Mean=0.542
  - price_sensitivity:  Mean=0.554
  - cuisine_consistency: Mean=0.762
  - age_group:          {'Millennial': 24226, 'Gen_X': 24176, 'Gen_Z': 17671, 'Boomer': 1559}


## Section 4: Item Features

**Features to create:**
- `popularity_rank` - Item popularity (1 = most popular)
- `avg_price` - Average selling price
- `item_order_count` - Total orders containing this item
- `category` - Item category (main, side, drink, dessert)

In [33]:
# ============================================================================
# ITEM FEATURES (FROM TRAIN SPLIT ONLY) - ENHANCED WITH NEW ATTRIBUTES
print("Computing item features from training data...")

# Use only TRAIN order items
item_stats = (
    train_order_items
    .groupby("item_id")
    .agg(
        item_order_count=("order_id", "nunique"),
        avg_price=("price", "mean"),
        category=("category", "first"),
        cuisine=("cuisine", "first"),
        is_vegetarian=("is_vegetarian", "first"),
        spice_level=("spice_level", "first"),
        price_tier=("price_tier", "first"),
        addon_rate=("addon_rate", "first"),
    )
    .reset_index()
)

# Popularity rank (1 = most popular)
item_stats["popularity_rank"] = (
    item_stats["item_order_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Normalize popularity score (0-1)
max_popularity = item_stats["item_order_count"].max()
item_stats["popularity_score"] = item_stats["item_order_count"] / max_popularity

# Normalize price (0-1)
max_price = item_stats["avg_price"].max()
item_stats["price_normalized"] = item_stats["avg_price"] / max_price

# Encode categorical attributes
item_stats["is_vegetarian_int"] = item_stats["is_vegetarian"].astype(int)

item_features = item_stats[[
    "item_id",
    "item_order_count",
    "avg_price",
    "popularity_rank",
    "popularity_score",
    "price_normalized",
    "category",
    "cuisine",
    "is_vegetarian",
    "spice_level",
    "price_tier",
    "addon_rate",
    "is_vegetarian_int",
]].copy()

print(f" Item features computed: {item_features.shape}")
print(f"\n New item attributes included:")
print(f"  - is_vegetarian: {item_features['is_vegetarian'].sum()} veg items")
print(f"  - spice_level: {item_features['spice_level'].nunique()} levels")
print(f"  - price_tier: {item_features['price_tier'].unique()}")
print(f"  - addon_rate: Mean={item_features['addon_rate'].mean():.3f}")
print(f"\nTop 10 popular items:")
print(item_features.nsmallest(10, "popularity_rank")[["item_id", "popularity_rank", "item_order_count", "category", "addon_rate"]])

Computing item features from training data...
 Item features computed: (2800, 13)

 New item attributes included:
  - is_vegetarian: 1544 veg items
  - spice_level: 3 levels
  - price_tier: <ArrowStringArray>
['standard', 'budget', 'premium']
Length: 3, dtype: str
  - addon_rate: Mean=0.367

Top 10 popular items:
      item_id  popularity_rank  item_order_count category  addon_rate
2269     2270                1               355     side      0.4590
2578     2579                2               346     side      0.4360
1634     1635                3               305    drink      0.5580
1628     1629                4               294    drink      0.4630
1612     1613                5               292    drink      0.4020
1003     1004                6               291  dessert      0.5240
2760     2761                7               290     side      0.3630
2262     2263                8               285     side      0.4670
1180     1181                9               277    dri

## Section 5: Context Features

**Features to create:**
- `hour` - Hour of day (0-23)
- `day_of_week` - Day of week (0-6)
- `is_weekend` - Weekend flag
- `season` - Season (Summer, Monsoon, Winter)
- `distance_km` - Delivery distance
- `delivery_fee` - Delivery cost
- `cart_size` - Items in cart
- `zone_type` - Delivery zone (CBD, Residential, Student)

In [34]:
# ============================================================================
# CONTEXT FEATURES (FROM ORDERS)
# ============================================================================
print("Extracting context features...")

# orders_enriched has duplicate columns from the pre-processing merge:
#   cart_size_x / cart_size_y, has_main_x / has_main_y, etc.
# Use the _x versions which reflect the original order composition.
context_features = orders[[
    "order_id",
    "user_id",
    "order_date",
    "distance_km",
    "delivery_fee",
    "cart_size_x",
    "has_main_x",
    "has_side_x",
    "has_drink_x",
    "has_dessert_x",
    "zone_type",
    "season",
    "tier",
    "city",
    "time_bucket"
]].copy()

# Rename to clean column names
context_features = context_features.rename(columns={
    "cart_size_x":    "cart_size",
    "has_main_x":    "has_main",
    "has_side_x":    "has_side",
    "has_drink_x":   "has_drink",
    "has_dessert_x": "has_dessert",
})
# Convert bool → int for downstream numerics
for col in ["has_main", "has_side", "has_drink", "has_dessert"]:
    context_features[col] = context_features[col].astype(int)

# Temporal features
context_features["hour"] = context_features["order_date"].dt.hour
context_features["day_of_week"] = context_features["order_date"].dt.dayofweek  # 0=Mon, 6=Sun
context_features["is_weekend"] = context_features["day_of_week"].isin([5, 6]).astype(int)
context_features["month"] = context_features["order_date"].dt.month

# Cyclic encoding of hour & day-of-week
context_features["hour_sin"] = np.sin(2 * np.pi * context_features["hour"] / 24)
context_features["hour_cos"] = np.cos(2 * np.pi * context_features["hour"] / 24)

context_features["dow_sin"] = np.sin(2 * np.pi * context_features["day_of_week"] / 7)
context_features["dow_cos"] = np.cos(2 * np.pi * context_features["day_of_week"] / 7)

# Normalize distance and delivery fee
context_features["distance_normalized"] = context_features["distance_km"] / context_features["distance_km"].max()
context_features["delivery_fee_normalized"] = context_features["delivery_fee"] / context_features["delivery_fee"].max()

# Log cart size (to handle skew)
context_features["cart_size_log"] = np.log1p(context_features["cart_size"])

print(f" Context features extracted: {context_features.shape}")
print(f"  Cart composition columns: has_main, has_side, has_drink, has_dessert")
print(f"  Temporal bucket: time_bucket ({context_features['time_bucket'].unique()})")
print(f"\nSample context:")
print(context_features.head())


Extracting context features...
 Context features extracted: (179364, 26)
  Cart composition columns: has_main, has_side, has_drink, has_dessert
  Temporal bucket: time_bucket (<ArrowStringArray>
['other']
Length: 1, dtype: str)

Sample context:
   order_id  user_id order_date  distance_km  delivery_fee  cart_size  \
0    873978    37324 2022-11-02       5.6400            40          1   
1    152962    33169 2019-02-24       0.8400             0          1   
2    557303    15951 2020-08-25       3.8200            40          4   
3    714027    12087 2020-05-04       3.0300            40          2   
4    757040    15946 2023-06-03       0.8900             0          1   

   has_main  has_side  has_drink  has_dessert zone_type   season  tier  \
0         0         0          1            0       CBD   Winter     2   
1         0         0          0            1       CBD   Winter     2   
2         1         1          1            1   Student  Monsoon     2   
3         0         

## Section 6: Interaction Features (Smart!)

**Strategy for interaction features:**
1. **Co-occurrence score:** How often item appears with cart items
2. **FP-Growth confidence:** Patterns from frequent itemset mining
3. **User-item affinity:** Did user buy similar items before?
4. **Category compatibility:** Natural pairings (main+side, main+drink)

In [35]:
# ============================================================================
# INTERACTION FEATURES (FROM TRAIN DATA)
# ============================================================================
print("Computing interaction features...")

# 1. BUILD CO-OCCURRENCE LOOKUP (from training data)
cooc_dict = {}
for _, row in cooccurrence.iterrows():
    item_i, item_j, count = row["item_i"], row["item_j"], row["count"]
    cooc_dict[(item_i, item_j)] = count
    cooc_dict[(item_j, item_i)] = count

# Normalize co-occurrence scores
max_cooc = max(cooc_dict.values()) if cooc_dict else 1
cooc_dict_norm = {k: v / max_cooc for k, v in cooc_dict.items()}

# 2. USER-ITEM AFFINITY (items user has bought in past)
user_purchased = (
    train_order_items
    .merge(train_orders[["order_id", "user_id"]], on="order_id")
    .groupby("user_id")["item_id"]
    .apply(set)
    .reset_index()
    .rename(columns={"item_id": "purchased_items"})
)

# 3. CATEGORY AFFINITY (what categories does user prefer)
user_category_prefs = (
    train_order_items
    .merge(train_orders[["order_id", "user_id"]], on="order_id")
    .groupby("user_id")["category"]
    .apply(lambda x: x.value_counts().to_dict())
    .reset_index()
    .rename(columns={"category": "category_counts"})
)

print(f" Co-occurrence index: {len(cooc_dict_norm):,} pairs")
print(f" User purchased items: {len(user_purchased):,} users")
print(f" User category preferences: {len(user_category_prefs):,} users")

Computing interaction features...
 Co-occurrence index: 237,390 pairs
 User purchased items: 41,733 users
 User category preferences: 166,932 users


## Section 6b: Cross-Features & Segment-Aware Interactions

**Cross-feature strategy:**
1. **Affordability:** User AOV vs Item Price (willingness to pay)
2. **Preference-Item Match:** User's veg% vs Item's veg status
3. **Context-Item Alignment:** Season/time effects on category demand
4. **Segment-Specific Co-occurrence:** Different patterns by tier/age/zone


In [36]:
# ============================================================================
# CROSS-FEATURES & SEGMENT-AWARE INTERACTIONS (FROM TRAIN DATA)
# ============================================================================
print("Computing cross-features and segment-aware interactions...")

# Build helper dictionaries for fast lookup
user_avg_aov = user_features.set_index("user_id")["avg_order_value"].to_dict()
user_veg_pref = user_features.set_index("user_id")["vegetarian_pct"].to_dict()
user_age_group = user_features.set_index("user_id")["age_group"].to_dict()
user_price_sens = user_features.set_index("user_id")["price_sensitivity"].to_dict()

item_price = item_features.set_index("item_id")["avg_price"].to_dict()
item_veg = item_features.set_index("item_id")["is_vegetarian_int"].to_dict()
item_addon_rate = item_features.set_index("item_id")["addon_rate"].to_dict()

# ============================================================================
# SEGMENT-AWARE CO-OCCURRENCE — vectorised, no iterrows, no per-order scan
# ============================================================================
print("  Building segment-aware co-occurrence matrices...")

# Join train order items with tier + season from train_orders in one pass
train_items_with_meta = (
    train_order_items
    .merge(train_orders[["order_id", "tier", "season"]], on="order_id", how="left")
)

segment_cooccurrence = defaultdict(lambda: defaultdict(int))

for (order_id, tier, season), group in train_items_with_meta.groupby(
    ["order_id", "tier", "season"]
):
    items = group["item_id"].tolist()
    seg_key = f"T{int(tier)}_S{season}"
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            pair = tuple(sorted([items[i], items[j]]))
            segment_cooccurrence[seg_key][pair] += 1

print(f"   Segment-aware co-occurrence: {len(segment_cooccurrence)} segments")
print(f"    Segments: {list(segment_cooccurrence.keys())}")
total_pairs = sum(len(v) for v in segment_cooccurrence.values())
print(f"    Total unique pairs across all segments: {total_pairs:,}")


Computing cross-features and segment-aware interactions...
  Building segment-aware co-occurrence matrices...
   Segment-aware co-occurrence: 9 segments
    Segments: ['T1_SWinter', 'T1_SMonsoon', 'T3_SMonsoon', 'T3_SSummer', 'T2_SMonsoon', 'T2_SSummer', 'T2_SWinter', 'T1_SSummer', 'T3_SWinter']
    Total unique pairs across all segments: 112,757


## Section 7: Pre-build Lookup Structures

Build all fast-lookup dicts from the already-computed interaction artefacts. These small
structures stay in memory for the entire pipeline; only one split's feature DataFrame
lives in RAM at a time.


In [37]:
import gc, joblib

# ============================================================================
# PRE-COMPUTE ALL LOOKUP STRUCTURES (small dicts — kept in memory throughout)
# ============================================================================
print("Pre-computing lookup structures...")

# item → co-occurrence neighbours (both directions already in cooc_dict_norm)
item_neighbours: dict = defaultdict(list)
for (item_i, item_j) in cooc_dict_norm:
    item_neighbours[item_i].append(item_j)

# order_id → set of item_ids  (positive example generation)
order_items_lookup: dict = (
    order_items.groupby("order_id")["item_id"].apply(set).to_dict()
)

# order_id → list of item_ids  (cooc_with_cart feature)
order_cart_lookup: dict = (
    order_items.groupby("order_id")["item_id"].apply(list).to_dict()
)

# order_id → distinct category count  (cart diversity feature)
order_category_count: dict = (
    order_items.groupby("order_id")["category"].nunique().to_dict()
)

# item → total co-occurrence strength (normalised)
from collections import defaultdict as _dd
item_cooc_total: dict = _dd(float)
for (item_i, item_j), score in cooc_dict_norm.items():
    item_cooc_total[item_i] += score
max_total_cooc = max(item_cooc_total.values()) if item_cooc_total else 1.0
item_cooc_total_norm = {k: v / max_total_cooc for k, v in item_cooc_total.items()}

# Normalise segment_cooccurrence counts per segment key
seg_cooc_norm: dict = {}
for seg_key, pair_dict in segment_cooccurrence.items():
    max_val = max(pair_dict.values()) if pair_dict else 1
    seg_cooc_norm[seg_key] = {pair: cnt / max_val for pair, cnt in pair_dict.items()}

all_items = set(item_features["item_id"].unique())
rng = np.random.default_rng(42)

print(f"   item_neighbours:      {len(item_neighbours):,} items")
print(f"   order_items_lookup:   {len(order_items_lookup):,} orders")
print(f"   order_cart_lookup:    {len(order_cart_lookup):,} orders")
print(f"   item_cooc_total_norm: {len(item_cooc_total_norm):,} items")
print(f"   seg_cooc_norm:        {len(seg_cooc_norm)} segments")
print("\n Pre-computation complete — ready for per-split chunked pipeline")


Pre-computing lookup structures...
   item_neighbours:      2,800 items
   order_items_lookup:   179,364 orders
   order_cart_lookup:    179,364 orders
   item_cooc_total_norm: 2,800 items
   seg_cooc_norm:        9 segments

 Pre-computation complete — ready for per-split chunked pipeline


## Section 8: Chunked Per-Split Feature Pipeline (16 GB RAM safe)

**Processes one split at a time — never holds all three in memory simultaneously.**

For each split (train → val → test):
1. Generate `(order_id, user_id, item_id, label)` examples
2. Merge user / item / context features
3. Compute all cross-features with **vectorised** `np.select` / `np.where` (no `.apply()`)
4. Impute with train medians; encode with fixed train vocabulary (no val/test column mismatch)
5. Scale with `StandardScaler` fitted on train only
6. Export to parquet, **then `del df; gc.collect()`**

Statistics learned from train (`season_cat_accept_rate`, medians, vocab, scaler) are
carried forward to val/test — no data leakage.


In [38]:

# ============================================================================
# CHUNKED PER-SPLIT PIPELINE  (16 GB RAM safe)
#   train → val → test processed one at a time, exported, then freed
#   Peak RAM ≈ one split's features at a time  (≈3× lower than before)
# ============================================================================

# State accumulated from train split and reused for val/test
season_cat_accept_rate: pd.DataFrame | None = None
FALLBACK_ACCEPT_RATE: float = 0.5
cat_vocabularies: dict | None = None
train_medians: dict | None = None
scaler = None
scale_cols: list | None = None

CATEGORICAL_COLS = [
    "preferred_cuisine", "category", "cuisine", "zone_type", "season",
    "spice_level", "price_tier", "age_group", "tenure_segment", "time_bucket"
]
BINARY_COLS = {
    "label", "is_weekend", "active_in_last_30d", "is_vegetarian_int",
    "is_member", "has_main", "has_side", "has_drink", "has_dessert",
}
SKIP_COLS = {"order_id", "user_id", "item_id", "label", "split", "order_date"} | BINARY_COLS

# Drop redundant user_id from context_features to avoid user_id_x/user_id_y after merge
context_features_no_uid = context_features.drop(columns=["user_id"], errors="ignore")

for split_name, split_orders in [
    ("train", train_orders),
    ("val",   val_orders),
    ("test",  test_orders),
]:
    print(f"\n{'='*60}")
    print(f"  SPLIT: {split_name.upper()}   ({len(split_orders):,} orders)")
    print(f"{'='*60}")

    # ── STEP 1: Build (order_id, user_id, item_id, label) rows ─────────────
    print("  [1/7] Generating examples...")
    rows = []
    for _, order in split_orders.iterrows():
        order_id = order["order_id"]
        user_id  = order["user_id"]
        items_in_order = order_items_lookup.get(order_id, set())
        if not items_in_order:
            continue

        for item_id in items_in_order:
            rows.append((order_id, user_id, item_id, 1))

        candidates: set = set()
        for cart_item in items_in_order:
            for nb in item_neighbours.get(cart_item, []):
                if nb not in items_in_order:
                    candidates.add(nb)
        for item_id in list(candidates)[:30]:
            rows.append((order_id, user_id, item_id, 0))

        true_negs = list(all_items - items_in_order - candidates)
        n_sample = min(5, len(true_negs))
        if n_sample > 0:
            for item_id in rng.choice(true_negs, size=n_sample, replace=False):
                rows.append((order_id, user_id, int(item_id), 0))

    df = pd.DataFrame(rows, columns=["order_id", "user_id", "item_id", "label"])
    pos = (df["label"] == 1).sum()
    print(f"       {len(df):,} examples  ({pos:,} pos, {pos/len(df)*100:.1f}%)")
    del rows; gc.collect()

    # ── STEP 2: Merge features ─────────────────────────────────────────────
    print("  [2/7] Merging user / item / context features...")
    df = df.merge(user_features,            on="user_id",  how="left")
    df = df.merge(item_features,            on="item_id",  how="left")
    df = df.merge(context_features_no_uid, on="order_id", how="left")

    # ── STEP 3: Vectorised cross-features (no .apply()) ────────────────────
    print("  [3/7] Computing cross-features (vectorised)...")

    df["affordability_ratio"] = (
        df["avg_order_value"] / df["avg_price"].clip(lower=50)
    ).fillna(0)

    df["veg_preference_match"] = (
        1 - (df["vegetarian_pct"] - df["is_vegetarian_int"]).abs()
    ).fillna(0.5)

    pt = df["price_tier"].fillna("mid")
    ps = df["price_sensitivity"].fillna(0.5)
    df["price_tier_affinity"] = np.select(
        [pt == "budget", pt == "premium"],
        [0.8 - ps * 0.2,  0.2 + ps * 0.2],
        default=0.5
    )

    df["tier_x_price"] = df["tier"] * df["avg_price"].fillna(0)

    # season × category accept rate: fit on train labels, apply to all
    if split_name == "train":
        season_cat_accept_rate = (
            df.groupby(["season", "category"])["label"]
            .mean()
            .rename("season_cat_accept_rate")
            .reset_index()
        )
        FALLBACK_ACCEPT_RATE = float(df["label"].mean())
    df = df.merge(season_cat_accept_rate, on=["season", "category"], how="left")
    df["season_cat_accept_rate"] = df["season_cat_accept_rate"].fillna(FALLBACK_ACCEPT_RATE)

    boost = np.where(
        (df["tier"] == 1) & (df["season"] == "Summer") & (df["category"] == "dessert"), 0.15,
        np.where((df["zone_type"] == "Student") & (df["category"] == "drink"), 0.20, 0.0)
    )
    df["segment_addon_score"] = (df["addon_rate"].fillna(0) + boost).clip(upper=0.99)

    age_drink_map = {"Gen_Z": 0.8, "Millennial": 0.65, "Gen_X": 0.45, "Boomer": 0.3}
    df["age_addon_affinity"] = np.where(
        df["category"] == "drink",
        df["age_group"].map(age_drink_map).fillna(0.5),
        0.5
    )

    df["cart_diversity_score"] = (
        df["order_id"].map(order_category_count).fillna(1) / 4.0
    )

    cat = df["category"]
    hm  = df["has_main"].astype(bool)
    df["item_complementarity"] = np.select(
        [
            (cat == "main")    &  hm,
            (cat == "main")    & ~hm,
            (cat == "side")    &  hm,
            (cat == "side")    & ~hm,
            (cat == "drink"),
            (cat == "dessert") &  hm,
            (cat == "dessert") & ~hm,
        ],
        [0.1, 0.9, 0.8, 0.3, 0.7, 0.6, 0.2],
        default=0.5
    )

    df["item_cooc_strength"] = df["item_id"].map(item_cooc_total_norm).fillna(0)

    # cooc_with_cart — per-row lookup, unavoidable but O(cart_size) per row
    oids = df["order_id"].tolist()
    iids = df["item_id"].tolist()
    cooc_arr = np.zeros(len(df), dtype=np.float32)
    for idx, (oid, iid) in enumerate(zip(oids, iids)):
        cart = order_cart_lookup.get(oid, [])
        if cart:
            cooc_arr[idx] = float(np.mean([cooc_dict_norm.get((iid, ci), 0.0) for ci in cart]))
    df["cooc_with_cart"] = cooc_arr

    # segment_cooc_score
    tiers   = df["tier"].tolist()
    seasons = df["season"].tolist()
    seg_arr = np.zeros(len(df), dtype=np.float32)
    for idx, (tier, season, oid, cand) in enumerate(zip(tiers, seasons, oids, iids)):
        seg_key  = f"T{tier}_S{season}"
        seg_dict = seg_cooc_norm.get(seg_key, {})
        if not seg_dict:
            continue
        cart = order_cart_lookup.get(oid, [])
        best = 0.0
        for ci in cart:
            s = seg_dict.get(tuple(sorted([cand, ci])), 0.0)
            if s > best:
                best = s
        seg_arr[idx] = best
    df["segment_cooc_score"] = seg_arr

    # ── STEP 4: Impute (medians from train only) ───────────────────────────
    print("  [4/7] Imputing missing values...")
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if split_name == "train":
        # Store medians for ALL numerical cols (not just those null in train).
        # Cold-start users in val/test have NaN for every user-feature column;
        # those cols have no nulls in train so the old `if .isnull().any()` guard
        # excluded them, leaving millions of nulls in val/test output.
        train_medians = {col: df[col].median() for col in numerical_cols}
    for col, med in train_medians.items():
        if col in df.columns:
            df[col] = df[col].fillna(med)
    for col in CATEGORICAL_COLS:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown")
    # Boolean flag columns (bool dtype is skipped by select_dtypes(include=np.number)
    # when pandas casts a bool-with-NaN to a nullable extension type).
    # Cold-start users produce NaN for is_member; default to 0 (not a member).
    for col in BINARY_COLS - {"label"}:
        if col in df.columns and df[col].isnull().any():
            df[col] = df[col].fillna(0).astype("int8")

    # ── STEP 5: One-hot encode with fixed train vocab ──────────────────────
    print("  [5/7] Encoding categoricals (fixed train vocab)...")
    if split_name == "train":
        cat_vocabularies = {
            col: sorted(df[col].dropna().unique().tolist())
            for col in CATEGORICAL_COLS if col in df.columns
        }
    for col, vocab in cat_vocabularies.items():
        if col not in df.columns:
            continue
        for val in vocab[1:]:   # drop_first equivalent: skip vocab[0]
            df[f"{col}_{val}"] = (df[col] == val).astype(np.int8)
        df = df.drop(columns=[col])

    # ── STEP 6: Scale (fit on train, apply to all) ─────────────────────────
    print("  [6/7] Scaling continuous features...")
    if split_name == "train":
        scale_cols = [
            col for col in df.columns
            if col not in SKIP_COLS
            and df[col].dtype in [np.float64, np.int64, np.float32, np.int32]
            and df[col].nunique() > 2
        ]
        scaler = StandardScaler()
        scaler.fit(df[scale_cols])
        joblib.dump(scaler, FEATURES_DIR / "feature_scaler.pkl")
        print(f"       ✓ Scaler fitted on {len(scale_cols)} cols & saved")

    df[scale_cols] = scaler.transform(df[scale_cols])
    df["split"] = split_name

    # ── STEP 7: Export immediately, then free ─────────────────────────────
    out_path = FEATURES_DIR / f"{split_name}_features.parquet"
    df.to_parquet(out_path, index=False)
    print(f"  [7/7] ✓ Saved {out_path.name}  —  {len(df):,} rows × {df.shape[1]} cols")

    del df, cooc_arr, seg_arr, oids, iids
    gc.collect()

print("\n" + "="*60)
print(" ALL SPLITS PROCESSED & EXPORTED")
print("="*60)



  SPLIT: TRAIN   (89,682 orders)
  [1/7] Generating examples...
       3,343,559 examples  (209,342 pos, 6.3%)
  [2/7] Merging user / item / context features...
  [3/7] Computing cross-features (vectorised)...
  [4/7] Imputing missing values...
  [5/7] Encoding categoricals (fixed train vocab)...
  [6/7] Scaling continuous features...
        Scaler fitted on 41 cols & saved
  [7/7] Saved train_features.parquet  —  6,358,947 rows × 86 cols

  SPLIT: VAL   (44,841 orders)
  [1/7] Generating examples...
       1,671,513 examples  (104,595 pos, 6.3%)
  [2/7] Merging user / item / context features...
  [3/7] Computing cross-features (vectorised)...
  [4/7] Imputing missing values...
  [5/7] Encoding categoricals (fixed train vocab)...
  [6/7] Scaling continuous features...
  [7/7] ✓ Saved val_features.parquet  —  2,536,369 rows × 86 cols

  SPLIT: TEST   (44,841 orders)
  [1/7] Generating examples...
       1,671,752 examples  (104,627 pos, 6.3%)
  [2/7] Merging user / item / context feat

## Section 9: Verify Scaler

The scaler was fitted on the train split inside the loop above and saved to
`data/features/feature_scaler.pkl`. This cell reloads it from disk to confirm
it was persisted correctly.


In [39]:
# ============================================================================
# VERIFY SCALER  (fitted on train inside the loop above and saved to disk)
# ============================================================================
scaler_path = FEATURES_DIR / "feature_scaler.pkl"
loaded_scaler = joblib.load(scaler_path)
print(f" Scaler loaded from {scaler_path}")
print(f"  Fitted on {len(loaded_scaler.mean_)} continuous features")
print(f"  Sample scale_cols: {', '.join(scale_cols[:6])}...")
print(f"  Binary flags excluded from scaling: {sorted(BINARY_COLS)}")


✓ Scaler loaded from ../data/features/feature_scaler.pkl
  Fitted on 41 continuous features
  Sample scale_cols: total_orders, avg_order_value, max_order_value, avg_subtotal, order_frequency_per_week, orders_last_7d...
  Binary flags excluded from scaling: ['active_in_last_30d', 'has_dessert', 'has_drink', 'has_main', 'has_side', 'is_member', 'is_vegetarian_int', 'is_weekend', 'label']


## Section 10: Validation Summary

Reads each split back from parquet (small random-access read) and confirms:
- Row counts and positive rates
- Feature presence (all key interaction features exist)
- Zero nulls across all splits
- All compliance checks


In [40]:
# ============================================================================
# VALIDATION SUMMARY  (reads from disk — no full-data copy in RAM)
print("Loading exported feature sets for validation...")
train_set = pd.read_parquet(FEATURES_DIR / "train_features.parquet")
val_set   = pd.read_parquet(FEATURES_DIR / "val_features.parquet")
test_set  = pd.read_parquet(FEATURES_DIR / "test_features.parquet")

print("\n" + "="*80)
print("FEATURE ENGINEERING SUMMARY ")
print("="*80)

print(f"\n DATASET STATISTICS")
for name, ds in [("Train", train_set), ("Val", val_set), ("Test", test_set)]:
    pos = (ds["label"] == 1).sum()
    print(f"  {name:5s}: {len(ds):,} examples | Positive: {pos:,} ({pos/len(ds)*100:.1f}%)")

print(f"\nFEATURE COUNT: {len(scale_cols)} scaled continuous + binary flags + one-hot dummies")
print(f"  Total columns in output: {train_set.shape[1]}")

print(f"\n USER COVERAGE")
print(f"  Train users: {train_set['user_id'].nunique():,}")
print(f"  Val users:   {val_set['user_id'].nunique():,}")
print(f"  Test users:  {test_set['user_id'].nunique():,}")

print(f"\n ITEM COVERAGE")
print(f"  Train items: {train_set['item_id'].nunique():,}")
print(f"  Val items:   {val_set['item_id'].nunique():,}")
print(f"  Test items:  {test_set['item_id'].nunique():,}")

print(f"\n DATA QUALITY")
for name, ds in [("train", train_set), ("val", val_set), ("test", test_set)]:
    nulls = ds.isnull().sum().sum()
    status = "✓" if nulls == 0 else f"✗ {nulls} nulls"
    print(f"  {status}  {name}")

print(f"\n KEY INTERACTION FEATURES")
for f_name in ["cooc_with_cart", "segment_cooc_score", "item_cooc_strength",
               "affordability_ratio", "tier_x_price", "season_cat_accept_rate",
               "item_complementarity", "cart_diversity_score",
               "orders_last_7d", "orders_last_30d", "max_order_value"]:
    status = "" if f_name in train_set.columns else "✗ MISSING"
    print(f"  {status}  {f_name}")

print(f"\n SEGMENT COVERAGE (One-hot encoded age groups)")
age_group_cols = [c for c in train_set.columns if c.startswith("age_group_")]
if age_group_cols:
    for col in sorted(age_group_cols):
        label = col.replace("age_group_", "")
        print(f"    {label:12s}: {int(train_set[col].sum()):,} train examples (encoded=1)")
else:
    print("    (age_group reference category absorbed into intercept)")

print(f"\n FEATURE ARTIFACTS SAVED")
for fname in ["train_features.parquet", "val_features.parquet",
              "test_features.parquet", "feature_scaler.pkl"]:
    p = FEATURES_DIR / fname
    status = "" if p.exists() else "✗ MISSING"
    print(f"  {status}  {fname}")

print(f"\n CSAO COMPLIANCE")
print(f"   Temporal split (order_id matched, not index)")
print(f"   Per-split chunked pipeline (16 GB RAM safe)")
print(f"   Fixed train vocabulary for one-hot encoding (no column mismatch)")
print(f"   season×category accept rate computed from train labels only")
print(f"   RFM metrics (orders_last_7d, orders_last_30d, max_order_value)")
print(f"   User behavioral segments (age, price sensitivity, veg preference)")
print(f"   Item attributes (addon_rate, vegetarian, spice, price_tier)")
print(f"   Cross-features (affordability, tier×price, season×category, complementarity)")
print(f"   Segment-aware co-occurrence (v2.0 core differentiator)")
print(f"   Context features (tier, zone, season, time_bucket, has_main/drink)")
print(f"   Binary flags excluded from StandardScaler")
print(f"   All .apply() replaced with vectorised np.select / np.where")

# Free loaded sets — validation is done
del train_set, val_set, test_set
gc.collect()

print("\n" + "="*80)
print(" FEATURE ENGINEERING PIPELINE COMPLETE")
print("="*80)
print("\nNext: Train ranking model with GBDT (LightGBM/XGBoost)")
print("  Input:  data/features/train_features.parquet")
print("  Target: label (1=positive add-on, 0=negative)")
print("  Metric: NDCG@8, Precision@8")


Loading exported feature sets for validation...

FEATURE ENGINEERING SUMMARY 

 DATASET STATISTICS
  Train: 6,358,947 examples | Positive: 397,711 (6.3%)
  Val  : 2,536,369 examples | Positive: 158,749 (6.3%)
  Test : 2,532,113 examples | Positive: 158,399 (6.3%)

 FEATURE COUNT: 41 scaled continuous + binary flags + one-hot dummies
  Total columns in output: 86

 USER COVERAGE
  Train users: 41,733
  Val users:   29,632
  Test users:  29,630

 ITEM COVERAGE
  Train items: 2,800
  Val items:   2,800
  Test items:  2,800

 DATA QUALITY
    train
   val
    test

 KEY INTERACTION FEATURES
    cooc_with_cart
    segment_cooc_score
    item_cooc_strength
    affordability_ratio
    tier_x_price
    season_cat_accept_rate
    item_complementarity
    cart_diversity_score
    orders_last_7d
    orders_last_30d
    max_order_value

 SEGMENT COVERAGE (One-hot encoded age groups)
    Gen_X       : 2,275,012 train examples (encoded=1)
    Gen_Z       : 1,657,453 train examples (encoded=1)
    Mi